In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@monarchazuredatalake.dfs.core.windows.net/products")

In [0]:
df.display()

In [0]:
df = df.drop("_rescued_data")


###### USER DEFIND Functions

- WR're creating a UDF (user-defined function) and registering it in Unity Catalog. That's the part the instructor means by "created in the catalogue."

- A normal Python function lives only in your notebook — when the notebook ends, it's gone, and no one else can use it. A UC-registered function is stored as a governed object inside a schema, right alongside your tables. That means:

- It persists — it's saved permanently, not tied to one notebook run. 
- It's reusable — any notebook, any SQL query, any other user with permission can call databricks_cata.bronze.discount_func(...).
- It's governed — same permission model as tables. You GRANT access to it. It shows up in Catalog Explorer under the schema, next to Tables and Volumes.


In [0]:

#temporary view to test the functions

df.createOrReplaceTempView("products_view")


######## **SQL FUNCTIONS**

In [0]:
%sql 
SHOW CATALOGS
-- to pick the catalog to attach the function

We need the catalog_name.schema.functon_Name(parameter)

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cata.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.95
     


In [0]:
%sql
select product_id, product_name,price, round(databricks_cata.bronze.discount_func(price),2) as discounted_price from products_view

###### Now I am confident with my sql query and the view results so now we can easily apply the function in the DataFrame

we want too create a new column, when you want to use the %sql function , we need expr()- to use the function similar as we used in the sql way

In [0]:
df = df.withColumn("discounted_price", expr("databricks_cata.bronze.discount_func(price)"))


In [0]:
df.display()

###### we can create python functions in python as well inside the SQL

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cata.bronze.upper_func(p_brand STRING)
RETURNS STRING 
LANGUAGE PYTHON
AS
$$
    return p_brand.upper()
$$


In [0]:
%sql
Select product_id,product_name, price, databricks_cata.bronze.upper_func(brand) as brand from products_view

##### Now finally writing the resultant table into the silver layer

In [0]:
df.write.format("delta")\
    .mode("append")\
        .option("path", "abfss://silver@monarchazuredatalake.dfs.core.windows.net/products")\
            .save()

## Now I am creating the regions silver layer part in this notebook only as there are no transformations required for this 

it is a static file which is not part of the star schema but, it needs to get passed to the gold layer

In [0]:
##regions is already present as delta table 
df_regions = spark.read.table("databricks_cata.default.regions")
df_regions.display() 

## remember the table is created in the default schema and not in the bronze schema

In [0]:
df_regions=df_regions.drop("_rescued_data")

In [0]:
df_regions.write.format("delta")\
    .mode("overwrite")\
        .save("abfss://silver@monarchazuredatalake.dfs.core.windows.net/regions")

#### Now lets finaly read all the dataframes before starting with the Gold layer

In [0]:
df_read = spark.read.format("delta").load("abfss://silver@monarchazuredatalake.dfs.core.windows.net/products")
df_read.display()